# 01. Data Loading and Merging

**Goal**: Load raw CSV files (Users, Telemetry, Death Events), remove PII, map death events to telemetry windows, and merge user metadata.

**Inputs**:
- `data/telemetry_phase_2.users.csv`
- `data/telemetry_phase_2.telemetries.csv`
- `data/telemetry_phase_2.deathevents.csv`

**Outputs**:
- `data/processed/merged_telemetry.csv` (Enriched with death counts and user data)
- `data/processed/merged_deathevents.csv` (Enriched with user data)

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import os

print("Libraries imported successfully.")

## 2. Define File Paths
We define the input paths for raw data and output paths for processed data.

In [ ]:
# File Paths
USERS_PATH = '../../data/telemetry_phase_2.users.csv'
TELEMETRY_PATH = '../../data/telemetry_phase_2.telemetries.csv'
DEATHEVENTS_PATH = '../../data/telemetry_phase_2.deathevents.csv'

# Output Paths
PROCESSED_DIR = '../../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

OUTPUT_TELEMETRY = os.path.join(PROCESSED_DIR, 'merged_telemetry.csv')
OUTPUT_DEATHEVENTS = os.path.join(PROCESSED_DIR, 'merged_deathevents.csv')

print(f"Input Users Path: {USERS_PATH}")
print(f"Input Telemetry Path: {TELEMETRY_PATH}")
print(f"Input Death Events Path: {DEATHEVENTS_PATH}")
print(f"Output Directory: {PROCESSED_DIR}")

## 3. Load Data
Loading the CSV files into Pandas DataFrames.

In [ ]:
print("Loading Users data...")
df_users = pd.read_csv(USERS_PATH)
print(f"Loaded Users: {df_users.shape}")
display(df_users.head(3))

In [ ]:
print("Loading Telemetry data...")
df_telemetry = pd.read_csv(TELEMETRY_PATH)
print(f"Loaded Telemetry: {df_telemetry.shape}")
# Optional: print columns to verify content
print("Telemetry Columns:", df_telemetry.columns.tolist())

In [ ]:
print("Loading Death Events data...")
df_deaths = pd.read_csv(DEATHEVENTS_PATH)
print(f"Loaded Death Events: {df_deaths.shape}")
display(df_deaths.head(3))

## 4. Remove PII from Users
We drop personally identifiable information like names and emails.

In [ ]:
# Columns to drop
pii_cols = ['firstName', 'lastName', 'email', 'password', 'username'] 

# Drop only columns that exist
cols_to_drop = [c for c in pii_cols if c in df_users.columns]
df_users_clean = df_users.drop(columns=cols_to_drop)

print("Columns removed:", cols_to_drop)
print(f"Cleaned Users Shape: {df_users_clean.shape}")
display(df_users_clean.head())

## 5. Map Death Events to Telemetry Windows

**Logic**:
1. Treat each telemetry row as a fixed 30-second behavioural window.
2. For each death event, find the telemetry window of the same user that occurs within the **next 30 seconds** after the death timestamp (`Death_Time <= Telemetry_Time < Death_Time + 30`).
3. Increment `death_count` on that telemetry row.

In [ ]:
# Filter users to those present in the users file
print("Filtering data to match valid user IDs...")
target_user_ids = set(df_users_clean['_id'].unique())

initial_tele_shape = df_telemetry.shape
initial_deaths_shape = df_deaths.shape

df_telemetry = df_telemetry[df_telemetry['userId'].isin(target_user_ids)].copy()
df_deaths = df_deaths[df_deaths['userId'].isin(target_user_ids)].copy()

print(f"Telemetry reduced from {initial_tele_shape} to {df_telemetry.shape}")
print(f"Deaths reduced from {initial_deaths_shape} to {df_deaths.shape}")

In [ ]:
# Ensure timestamps are numeric
print("Converting timestamps to numeric...")
df_telemetry['timestamp'] = pd.to_numeric(df_telemetry['timestamp'], errors='coerce')
df_deaths['timestamp'] = pd.to_numeric(df_deaths['timestamp'], errors='coerce')

# Initialize death_count
df_telemetry['death_count'] = 0

print("Timestamps ready and death_count initialized to 0.")

In [ ]:
# Sort for efficient searching 
print("Sorting dataframes by userId and timestamp...")
df_telemetry.sort_values(by=['userId', 'timestamp'], inplace=True)
df_deaths.sort_values(by=['userId', 'timestamp'], inplace=True)
print("Sorting complete.")

In [ ]:
print("Mapping deaths to telemetry windows...")
count_mapped = 0

# Group telemetry by user for faster lookup
telemetry_grouped = df_telemetry.groupby('userId')

total_death_events = len(df_deaths)
print(f"Total death events to process: {total_death_events}")

for user_id, user_deaths in df_deaths.groupby('userId'):
    if user_id not in telemetry_grouped.groups:
        continue
    
    # Get user's telemetry indices in the main DF
    user_tele_indices = telemetry_grouped.groups[user_id]
    user_tele_times = df_telemetry.loc[user_tele_indices, 'timestamp'].values
    
    for death_time in user_deaths['timestamp']:
        if pd.isna(death_time):
             continue
             
        # Condition: Death_Time <= Tele_Time < Death_Time + 30
        mask = (user_tele_times >= death_time) & (user_tele_times < death_time + 30)
        
        matched_indices_local = np.where(mask)[0]
        
        if len(matched_indices_local) > 0:
            # Increment count on the first valid window found
            match_idx = user_tele_indices[matched_indices_local[0]]
            df_telemetry.at[match_idx, 'death_count'] += 1
            count_mapped += 1

print(f"DONE: Mapped {count_mapped} / {total_death_events} death events to telemetry windows.")

## 6. Merge User Data into Telemetry and Death Events

In [ ]:
print("Merging User data into Telemetry...")
# Assuming Users table has '_id' which corresponds to 'userId' in others
user_key = '_id' if '_id' in df_users_clean.columns else 'userId'
telemetry_key = 'userId'
death_key = 'userId'

# Merge Telemetry
df_telemetry_merged = pd.merge(
    df_telemetry,
    df_users_clean,
    left_on=telemetry_key,
    right_on=user_key,
    how='left'
)
print(f"Merged Telemetry Shape: {df_telemetry_merged.shape}")

In [ ]:
print("Merging User data into Death Events...")
# Merge Death Events
df_deaths_merged = pd.merge(
    df_deaths,
    df_users_clean,
    left_on=death_key,
    right_on=user_key,
    how='left'
)
print(f"Merged Death Events Shape: {df_deaths_merged.shape}")

## 7. Save Processed Data

In [ ]:
print(f"Saving processed files to {PROCESSED_DIR}...")
df_telemetry_merged.to_csv(OUTPUT_TELEMETRY, index=False)
df_deaths_merged.to_csv(OUTPUT_DEATHEVENTS, index=False)

print(f"Saved merged telemetry to: {OUTPUT_TELEMETRY}")
print(f"Saved merged deaths to: {OUTPUT_DEATHEVENTS}")
print("Data loading and merging process completed successfully.")